# Experiment Notebook: Unified Benchmark Orchestrator

Goal: run a broad policy/model benchmark and export a consolidated hard-slice leaderboard with confidence intervals.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str((Path.cwd().parent / "src").resolve()))
from ml_engine import research_upgrade_lab as lab

lab.set_seed(42)
df = lab.load_clean_dataset(Path.cwd())
weights = {"carbon": 0.35, "jct": 0.20, "tail": 0.25, "preempt": 0.10, "starvation": 0.10}

# Train diverse policy forms
q_domain = lab.train_q_domain_randomized(df=df, episodes=170, reward_weights=weights, hard_bias=0.60)
q_hard = lab.train_q_domain_randomized(df=df, episodes=170, reward_weights=weights, hard_bias=0.85)

demo_counts = lab.collect_srtf_demo_counts(df=df, episodes=80)
q_init = demo_counts / np.maximum(1.0, demo_counts.sum(axis=-1, keepdims=True))
q_double = lab.train_double_q(df=df, episodes=210, reward_weights=weights, q_init=q_init)

policy_tables = {
    "fcfs": None,
    "carbon": None,
    "srtf": None,
    "q_domain": q_domain,
    "q_hard": q_hard,
    "q_double": q_double,
}

# 30-seed evaluation for report-grade robustness
detail = lab.evaluate_policies(
    df=df,
    policy_tables=policy_tables,
    seeds=list(range(30)),
    noises=[15.0, 20.0],
    congestions=["moderate", "high"],
    capacities=[6, 8],
    reward_weights=weights,
)

scored = lab.compute_overall_score_robust(detail, weights)
hard = lab.summarize_hard_slice(scored)

display(hard.round(4))

save_dir = Path("../data")
save_dir.mkdir(parents=True, exist_ok=True)
scored.to_csv(save_dir / "unified_benchmark_scored_detail.csv", index=False)
hard.to_csv(save_dir / "unified_benchmark_hard_slice.csv", index=False)
print("Saved:", save_dir / "unified_benchmark_scored_detail.csv")
print("Saved:", save_dir / "unified_benchmark_hard_slice.csv")

,policy,noise_pct,congestion,capacity,n,overall_score_mean,overall_score_std,carbon_per_completed_job_mean,avg_jct_mean,tail_jct_mean,jobs_completed_mean,overall_score_ci95
44,srtf,20.0,high,6,30,0.1164,0.0103,0.2802,1.0578,4.8854,212.9667,0.0037
20,q_domain,20.0,high,6,30,0.3185,0.0557,0.2893,2.5923,8.2989,198.0333,0.0199
36,q_hard,20.0,high,6,30,0.4332,0.0888,0.3658,2.0867,8.4735,161.9333,0.0318
28,q_double,20.0,high,6,30,0.5393,0.0877,0.3880,2.6734,8.5400,151.2667,0.0314
12,fcfs,20.0,high,6,30,0.7669,0.0457,0.4904,4.3226,10.6622,119.5000,0.0163
4,carbon,20.0,high,6,30,0.7986,0.0574,0.4383,3.8725,11.4560,127.2333,0.0205


Saved: ..\data\unified_benchmark_scored_detail.csv
Saved: ..\data\unified_benchmark_hard_slice.csv
